Import Libraries

In [ ]:
import jax
import flax
import optax
from jax import lax, random, numpy as jnp
from jax import random, vmap, hessian, jacfwd, jit, value_and_grad
from jax import config
from flax import linen as nn

import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors

import os
from tqdm.auto import tqdm

#os.environ['CUDA_VISIBLE_DEVICES'] = '1'
config.update("jax_enable_x64", True)
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'cuda_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'true' # default is true, 90% of GPU VRAM preallocated


In [ ]:
n_gpu = len(jax.devices())
jax.devices()

In [ ]:
# For Google Collab
# DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/"

# For PC
DATA_DIR = Path.cwd().parent.parent / "data"

In [ ]:
DATA_FILE = DATA_DIR / "data_T20_S30_G50.dat"

dat = np.loadtxt(DATA_FILE, skiprows=1, delimiter=',') # data on CPU
print (dat.shape)

# swap header for non reduced dataset as it was not labelled properly
if DATA_FILE == DATA_DIR / "data_T20_S30_G50.dat":
    dat[:, [0, 1]] = dat[:, [1, 0]]

# crop to focus on area of interest in the grid
# dat = dat[(dat[:,1] > 150) & (dat[:,1] < 350)] # selects all rows where the y-coordinate falls strictly between 150 and 350

# Calculate dimensions
nx = len(np.unique(dat[:, 0])) # take every row in x col 
ny = len(np.unique(dat[:, 1])) # take every row in y col

print(dat.shape)
print(f"Grid dimensions: nx={nx}, ny={ny}")
print(np.unique(dat[:,0]).shape, np.unique(dat[:,1]).shape) # unique coordinate values along each individual axis

x_l, x_u, y_l, y_u = np.unique(dat[:,0]).min(), np.unique(dat[:,0]).max(), np.unique(dat[:,1]).min(), np.unique(dat[:,1]).max()
ext = [x_l, x_u, y_l, y_u] # plot boundary
print(ext)

In [ ]:
"""
Plot the system with boundaries as the perimeter of the plot
"""
# Source
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1,2,1)
s_plot = dat[:,2].reshape(nx, ny).T
mesh2 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh2); plt.xlabel('x'); plt.ylabel('y')
plt.title('Source', fontsize='x-large')

# Solution
ax1 = fig.add_subplot(1,2,2)
s_plot = dat[:,3].reshape(nx, ny).T
mesh2 = plt.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
plt.colorbar(mesh2); plt.xlabel('x'); plt.ylabel('y')
plt.title('Solution', fontsize='x-large')

In [ ]:
dat = jnp.array(dat) 
x, y, s, u_sim = dat[:,0].reshape(-1, 1), dat[:,1].reshape(-1, 1), dat[:,2].reshape(-1, 1), dat[:,3].reshape(-1, 1)

# Find boundaries 
bc_n = ((y == y_l) | (y == y_u)).flatten()
bc_pl, bc_pu = ((x == x_l)).flatten(), ((x == x_u)).flatten()

# Boundary points for mini-batching 
x_n, y_n = x[bc_n], y[bc_n]
x_pl, y_pl = x[bc_pl], y[bc_pl]
x_pu, y_pu = x[bc_pu], y[bc_pu]

In [ ]:
M = 256 
hidden_layers = [128, 128, 128, 128] 
total_features = (2 * M) + sum(hidden_layers) 
chunk_size = 256 

sigma = 0.5 # Baseline sigma
rff_key = jax.random.PRNGKey(99)
B_matrix = jax.random.normal(rff_key, (2, M)) * sigma 

class FeatureExtractor(nn.Module):
    hidden_layers: list
    B_matrix: jnp.ndarray 
    
    @nn.compact
    def __call__(self, x_in, y_in):    
        self.param('raw_lambda', nn.initializers.constant(-3.0), (1,)) 
        
        v = jnp.concatenate([x_in, y_in])

        proj = 2.0 * jnp.pi * jnp.dot(v, self.B_matrix)
        h_rff = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)])
        all_features = [h_rff]

        h = h_rff
        for size in self.hidden_layers:
            h = nn.Dense(size, kernel_init=nn.initializers.he_normal())(h)
            h = nn.tanh(h) 
            all_features.append(h)
            
        combined_h = jnp.concatenate(all_features)
            
        return combined_h
    
model = FeatureExtractor(hidden_layers=hidden_layers, B_matrix=B_matrix)

# Initialize parameters
key = jax.random.PRNGKey(0)
dummy_x = jnp.array([0.0])
dummy_y = jnp.array([0.0])
params = model.init(key, dummy_x, dummy_y)

In [ ]:
def get_f(params, x_val, y_val):
    return model.apply(params, x_val, y_val)

def get_f_dir(params, x_val, y_val):
    f = get_f(params, x_val, y_val)
    
    # Second-order derivatives
    f_xx = jacfwd(jacfwd(get_f, argnums=1), argnums=1)(params, x_val, y_val)
    f_yy = jacfwd(jacfwd(get_f, argnums=2), argnums=2)(params, x_val, y_val)
    
    f_xx = jnp.squeeze(f_xx, axis=(-1, -2))
    f_yy = jnp.squeeze(f_yy, axis=(-1, -2))

    return f_xx, f_yy

def get_f_chunk(params, x_val, y_val, start_idx, chunk_size):
    f = model.apply(params, x_val, y_val)
    return lax.dynamic_slice(f, (start_idx,), (chunk_size,))

def get_f_dir_chunk(params, x_val, y_val, start_idx, chunk_size):
    f_xx_chunk = jacfwd(jacfwd(get_f_chunk, argnums=1), argnums=1)(params, x_val, y_val, start_idx, chunk_size)
    f_yy_chunk = jacfwd(jacfwd(get_f_chunk, argnums=2), argnums=2)(params, x_val, y_val, start_idx, chunk_size)
    
    f_xx_chunk = jnp.squeeze(f_xx_chunk, axis=(-1, -2))
    f_yy_chunk = jnp.squeeze(f_yy_chunk, axis=(-1, -2))
    
    return f_xx_chunk, f_yy_chunk

def get_u(params, x_val, y_val, w_current):
    f = model.apply(params, x_val, y_val)
    return jnp.dot(f, w_current) 

def get_u_dir(params, x_val, y_val, w_current):
    u_xx = jacfwd(jacfwd(get_u, argnums=1), argnums=1)(params, x_val, y_val, w_current)
    u_yy = jacfwd(jacfwd(get_u, argnums=2), argnums=2)(params, x_val, y_val, w_current)
    
    u_xx = jnp.squeeze(u_xx, axis=(-1, -2))
    u_yy = jnp.squeeze(u_yy, axis=(-1, -2))
    
    return u_xx, u_yy

f_spatial_vmap = vmap(get_f, in_axes=(None, 0, 0))
f_dir_spatial_vmap = vmap(get_f_dir, in_axes=(None, 0, 0))
f_chunk_spatial_vmap = vmap(get_f_chunk, in_axes=(None, 0, 0, None, None))
f_dir_chunk_spatial_vmap = vmap(get_f_dir_chunk, in_axes=(None, 0, 0, None, None))
u_dir_spatial_vmap = vmap(get_u_dir, in_axes=(None, 0, 0, None))

In [ ]:
def get_f_y_chunk(params, x_val, y_val, start_idx, chunk_size):
    f_y_chunk = jacfwd(get_f_chunk, argnums=2)(params, x_val, y_val, start_idx, chunk_size)
    return jnp.squeeze(f_y_chunk, axis=-1)

def get_u_y(params, x_val, y_val, w_current):
    u_y = jacfwd(get_u, argnums=2)(params, x_val, y_val, w_current)
    return jnp.squeeze(u_y, axis=-1)

f_y_chunk_spatial_vmap = vmap(get_f_y_chunk, in_axes=(None, 0, 0, None, None))
u_y_spatial_vmap = vmap(get_u_y, in_axes=(None, 0, 0, None))

In [ ]:
optimizer = optax.adamw(learning_rate=1e-4, weight_decay=1e-4)

def compute_loss(params, w_current, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu):
    u_xx, u_yy = u_dir_spatial_vmap(params, x_pde, y_pde, w_current)
    laplacian_u = u_xx + u_yy
    pde_residual = laplacian_u - s_pde 
    loss_pde = jnp.mean(pde_residual ** 2)
    
    u_y_n = u_y_spatial_vmap(params, x_n, y_n, w_current)
    loss_n = jnp.mean(u_y_n ** 2) * 1000.0
    
    u_pl = f_spatial_vmap(params, x_pl, y_pl) @ w_current
    u_pu = f_spatial_vmap(params, x_pu, y_pu) @ w_current
    loss_p = jnp.mean((u_pl - u_pu) ** 2) * 1000.0
    
    return loss_pde + loss_n + loss_p

In [ ]:
kaczmarz_sweeps = 1
kaczmarz_alpha = 1e-4 

In [ ]:
def kaczmarz_inner_update(params, w_k, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size, alpha, tik_reg):
    b_pde = s_pde
    b_n = jnp.zeros((x_n.shape[0], 1))
    b_p = jnp.zeros((x_pl.shape[0], 1))
    
    b_block = jnp.vstack([b_pde, b_n, b_p])
    N_rows = b_block.shape[0]
    
    num_chunks = total_features // chunk_size
    chunk_indices = jnp.arange(num_chunks) * chunk_size
    
    def build_gram_step(carry, start_idx):
        Aw_acc, G_acc = carry
        
        f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, chunk_size)
        A_pde_chunk = f_xx + f_yy
        
        A_n_chunk = f_y_chunk_spatial_vmap(params, x_n, y_n, start_idx, chunk_size) * jnp.sqrt(1000.0)
        
        f_pl = f_chunk_spatial_vmap(params, x_pl, y_pl, start_idx, chunk_size)
        f_pu = f_chunk_spatial_vmap(params, x_pu, y_pu, start_idx, chunk_size)
        A_p_chunk = (f_pl - f_pu) * jnp.sqrt(1000.0)
        
        A_i = jnp.vstack([A_pde_chunk, A_n_chunk, A_p_chunk])
        w_i = jax.lax.dynamic_slice(w_k, (start_idx, 0), (chunk_size, 1))
        
        Aw_acc += A_i @ w_i
        G_acc += A_i @ A_i.T
        
        return (Aw_acc, G_acc), None
        
    init_carry = (jnp.zeros((N_rows, 1)), jnp.zeros((N_rows, N_rows)))
    (Aw, G), _ = jax.lax.scan(build_gram_step, init_carry, chunk_indices)
    
    I = jnp.eye(N_rows)
    z = jnp.linalg.solve(G + tik_reg * I, Aw - b_block)
    
    def update_w_step(carry, start_idx):
        f_xx, f_yy = f_dir_chunk_spatial_vmap(params, x_pde, y_pde, start_idx, chunk_size)
        A_pde_chunk = f_xx + f_yy
        
        A_n_chunk = f_y_chunk_spatial_vmap(params, x_n, y_n, start_idx, chunk_size) * jnp.sqrt(1000.0)
        
        f_pl = f_chunk_spatial_vmap(params, x_pl, y_pl, start_idx, chunk_size)
        f_pu = f_chunk_spatial_vmap(params, x_pu, y_pu, start_idx, chunk_size)
        A_p_chunk = (f_pl - f_pu) * jnp.sqrt(1000.0)
        
        A_i = jnp.vstack([A_pde_chunk, A_n_chunk, A_p_chunk])
        w_i = jax.lax.dynamic_slice(w_k, (start_idx, 0), (chunk_size, 1))
        w_i_new = w_i - alpha * (A_i.T @ z)
        
        return carry, w_i_new
        
    _, w_new_chunks = jax.lax.scan(update_w_step, None, chunk_indices)
    w_new = w_new_chunks.reshape((total_features, 1))
    
    return w_new

In [ ]:
@jax.jit(static_argnames=['total_features', 'chunk_size', 'num_sweeps'])
def update_kaczmarz_sweep(params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size, alpha, tik_reg, num_sweeps):
    
    def sweep_step(w_carry, _):
        w_next = kaczmarz_inner_update(
            params, w_carry, x_pde, y_pde, s_pde, 
            x_n, y_n, x_pl, y_pl, x_pu, y_pu, 
            total_features, chunk_size, alpha, tik_reg
        )
        return w_next, None

    w_init = jnp.zeros((total_features, 1))
    w_final, _ = jax.lax.scan(sweep_step, w_init, jnp.arange(num_sweeps))
    
    return w_final


def full_forward_and_loss(params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size):
    raw_val = params['params']['raw_lambda'][0]
    tik_reg = 10 ** (jnp.tanh(raw_val) * 3 - 2)
    
    # Get analytical linear weights for the current spatial batch
    w_current = update_kaczmarz_sweep(
        params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, 
        total_features, chunk_size, kaczmarz_alpha, tik_reg, kaczmarz_sweeps
    )
    
    # Vital for saving VRAM - prevents gradient tracking through Kaczmarz
    w_current = jax.lax.stop_gradient(w_current)
    
    loss = compute_loss(params, w_current, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu)
    
    return loss, tik_reg

loss_grad_fn = jax.jit(
    value_and_grad(full_forward_and_loss, argnums=0, has_aux=True), 
    static_argnames=['total_features', 'chunk_size']
)

@jax.jit(static_argnames=['total_features', 'chunk_size'])
def update_network(params, opt_state, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size):
    
    (loss, current_lambda), grads = loss_grad_fn(
        params, x_pde, y_pde, s_pde, x_n, y_n, x_pl, y_pl, x_pu, y_pu, total_features, chunk_size
    )
    
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    new_params = optax.apply_updates(params, updates)
    
    return new_params, opt_state, loss, current_lambda

In [ ]:
epochs = 3
spatial_batch_size = 1000
bc_batch_size = 500

opt_state = optimizer.init(params)
rng = jax.random.PRNGKey(42)

# Ensure our dataset arrays exist in memory
total_points = x.shape[0]
total_n_points = x_n.shape[0]
total_p_points = x_pl.shape[0]

# Initial shuffling
rng, key_pde, key_n, key_p = jax.random.split(rng, 4)
idx_pde = jax.random.permutation(key_pde, total_points)
idx_n = jax.random.permutation(key_n, total_n_points)
idx_p = jax.random.permutation(key_p, total_p_points)

ptr_pde, ptr_n, ptr_p = 0, 0, 0

for epoch in range(epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    # One epoch = one full pass over the internal PDE points
    with tqdm(total=total_points // spatial_batch_size, desc=f"Epoch {epoch+1}/{epochs}") as pbar:
        while ptr_pde + spatial_batch_size <= total_points:
            
            # 1. Sample PDE internal points
            batch_idx = idx_pde[ptr_pde : ptr_pde + spatial_batch_size]
            x_pde_chunk = x[batch_idx]
            y_pde_chunk = y[batch_idx]
            s_pde_chunk = s[batch_idx]
            ptr_pde += spatial_batch_size
            
            # 2. Sample Neumann Boundary points
            if ptr_n + bc_batch_size > total_n_points:
                rng, key_n = jax.random.split(rng)
                idx_n = jax.random.permutation(key_n, total_n_points)
                ptr_n = 0
            
            # Handle edge case where total boundary points < requested batch size
            current_n_size = min(bc_batch_size, total_n_points)
            n_batch_idx = idx_n[ptr_n : ptr_n + current_n_size]
            x_n_chunk = x_n[n_batch_idx]
            y_n_chunk = y_n[n_batch_idx]
            ptr_n += current_n_size
            
            # 3. Sample Periodic Boundary points
            if ptr_p + bc_batch_size > total_p_points:
                rng, key_p = jax.random.split(rng)
                idx_p = jax.random.permutation(key_p, total_p_points)
                ptr_p = 0
            
            current_p_size = min(bc_batch_size, total_p_points)
            p_batch_idx = idx_p[ptr_p : ptr_p + current_p_size]
            x_pl_chunk = x_pl[p_batch_idx]
            y_pl_chunk = y_pl[p_batch_idx]
            x_pu_chunk = x_pu[p_batch_idx]
            y_pu_chunk = y_pu[p_batch_idx]
            ptr_p += current_p_size
            
            # 4. Network Update Step
            params, opt_state, loss, current_lambda = update_network(
                params, opt_state, 
                x_pde_chunk, y_pde_chunk, s_pde_chunk, 
                x_n_chunk, y_n_chunk, 
                x_pl_chunk, y_pl_chunk, x_pu_chunk, y_pu_chunk, 
                total_features, chunk_size
            )
            
            epoch_loss += loss
            num_batches += 1
            pbar.update(1)
            
    # Reshuffle PDE points for the next epoch
    rng, key_pde = jax.random.split(rng)
    idx_pde = jax.random.permutation(key_pde, total_points)
    ptr_pde = 0
            
    avg_train_loss = epoch_loss / max(1, num_batches)
    print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4e} | TikReg: {current_lambda:.2e}")

In [ ]:
# Grab 1000 random points across the whole grid to solve for the final analytical w
rng_plot = jax.random.PRNGKey(1)
plot_spatial_idx = jax.random.choice(rng_plot, jnp.arange(nx * ny), shape=(1000,), replace=False)

raw_val = params['params']['raw_lambda'][0]
learned_lambda = 10 ** (jnp.tanh(raw_val) * 3 - 2)

# Solve for w using Kaczmarz sweeps
w_plot = update_kaczmarz_sweep(
    params, 
    x[plot_spatial_idx], y[plot_spatial_idx], s[plot_spatial_idx], 
    x_n[:bc_batch_size], y_n[:bc_batch_size],  # Grab a chunk of boundaries
    x_pl[:bc_batch_size], y_pl[:bc_batch_size], x_pu[:bc_batch_size], y_pu[:bc_batch_size], 
    total_features, chunk_size, kaczmarz_alpha, learned_lambda, kaczmarz_sweeps
)

# Evaluate the full grid
eval_chunk_size = 10000
u_list = []

print("Evaluating full grid in chunks to save memory...")
for i in range(0, x.shape[0], eval_chunk_size):
    x_c = x[i : i + eval_chunk_size]
    y_c = y[i : i + eval_chunk_size]
    
    # Calculate features just for this chunk
    f_chunk = f_spatial_vmap(params, x_c, y_c)
    
    # Do the dot product immediately so we only store the (10000, 1) result
    u_chunk = f_chunk @ w_plot
    
    # Force JAX to execute and free up the VRAM before the next loop
    u_chunk.block_until_ready() 
    u_list.append(u_chunk)

u_pred = jnp.vstack(u_list)
u_list.clear() # Free up the python list memory

# Calculate Metrics
mse = jnp.mean((u_sim - u_pred)**2)
mae = jnp.mean(jnp.abs(u_sim - u_pred))
rl2 = jnp.linalg.norm(u_sim - u_pred) / jnp.linalg.norm(u_sim)

print(f'MSE = {mse:.2e} | MAE = {mae:.2e} | RL2 = {rl2:.2e}')

# Plotting
s_plot = s.reshape(nx, ny).T
u_sim_plot = u_sim.reshape(nx, ny).T 
u_pred_plot = u_pred.reshape(nx, ny).T

ext = [x_l, x_u, y_l, y_u]
fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(1, 3, 1)
mesh1 = ax1.imshow(s_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh1, ax=ax1)
ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_title('Source (s)', fontsize='x-large')

ax2 = fig.add_subplot(1, 3, 2)
mesh2 = ax2.imshow(u_sim_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh2, ax=ax2)
ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.set_title('Ground Truth Simulation (u)', fontsize='x-large')

ax3 = fig.add_subplot(1, 3, 3)
mesh3 = ax3.imshow(u_pred_plot, interpolation='bilinear', origin='lower', cmap='rainbow', extent=ext, aspect=1.)
fig.colorbar(mesh3, ax=ax3)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_title('RFF+MLP Prediction', fontsize='x-large')

plt.tight_layout()
plt.show()